# Eval Metric and Runtime Deltas

This notebook is intentionally split into stages:

1. Extract eval metrics from `eval_results/`
2. Inspect the parsed eval table locally
3. Query W&B Local for matching training runs and runtimes
4. Join both sources
5. Compute baseline-relative metric deltas and runtime percent deltas using `fixed_k_max`


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import wandb


In [2]:
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().resolve(), Path(".").resolve(), Path("..").resolve()]

    try:
        candidates.extend(Path(__file__).resolve().parents)
    except NameError:
        pass

    notebook_hint = Path("notebooks/eval_metric_runtime_deltas.ipynb").resolve()
    candidates.extend(notebook_hint.parents)

    for candidate in candidates:
        if (candidate / "eval_results").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        f"Could not locate repo root from cwd={Path.cwd()}"
    )


ROOT = resolve_repo_root()
EVAL_ROOT = ROOT / "eval_results"
OUTPUT_CSV = ROOT / "notebooks" / "eval_metric_runtime_deltas.csv"

WANDB_BASE_URL = os.environ.get("WANDB_BASE_URL", "http://localhost:8080")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "sebi")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "routing_curriculum")

METRIC_KEYS = {
    "arc": "accuracy",
    "sciq": "accuracy",
    "gsm8k": "accuracy_extracted_answer",
}

EVAL_VARIANT_PRIORITY = {
    "best": 0,
    "final": 1,
    "default": 2,
}

CURRICULUM_DISPLAY_ORDER = [
    "fixed_k_max",
    "fixed_k_1",
    "linear_k_1_to_topk",
    "frontloaded",
    "backloaded",
    "warmup",
    "jump_warmup",
    "linear_mid_start"
]

ROOT, EVAL_ROOT, OUTPUT_CSV

(PosixPath('/home/sebi/disertatie'),
 PosixPath('/home/sebi/disertatie/eval_results'),
 PosixPath('/home/sebi/disertatie/notebooks/eval_metric_runtime_deltas.csv'))

## 1. Extract Eval Results Only

This section only reads local files under `eval_results/`. No W&B calls happen yet.


In [3]:
def metric_key_for(metrics):
    benchmark = metrics["benchmark"]
    metric_key = METRIC_KEYS.get(benchmark)
    if metric_key is None or metric_key not in metrics:
        raise KeyError(
            f"Unsupported benchmark metric for {benchmark}: {sorted(metrics.keys())}"
        )
    return metric_key


def parse_run_name(run_name, benchmark):
    if run_name.startswith("base__"):
        return None

    base_run_name, separator, eval_variant = run_name.partition("__")
    if separator and eval_variant != "final":
        return None

    normalized_run_name = base_run_name if separator else run_name
    split_token = f"_{benchmark}_"
    if split_token not in normalized_run_name:
        return None

    model_family, suffix = normalized_run_name.split(split_token, 1)
    curriculum = suffix[:-5] if suffix.endswith("_lora") else suffix
    return model_family, curriculum


EVAL_COLUMNS = [
    "benchmark",
    "model_family",
    "curriculum",
    "run_name",
    "metric_name",
    "metric_value",
    "num_examples",
    "checkpoint_path",
    "model_id",
    "metrics_path",
]


def load_eval_table():
    rows = []

    for metrics_path in sorted(EVAL_ROOT.glob("*/*/metrics.json")):
        metrics = json.loads(metrics_path.read_text())
        run_name = metrics_path.parent.name
        benchmark = metrics["benchmark"]
        parsed = parse_run_name(run_name, benchmark)

        if parsed is None:
            continue

        model_family, curriculum = parsed
        metric_key = metric_key_for(metrics)

        rows.append(
            {
                "benchmark": benchmark,
                "model_family": model_family,
                "curriculum": curriculum,
                "run_name": run_name,
                "metric_name": metric_key,
                "metric_value": metrics[metric_key],
                "num_examples": metrics.get("num_examples"),
                "checkpoint_path": metrics.get("checkpoint_path"),
                "model_id": metrics.get("model_id"),
                "metrics_path": str(metrics_path),
            }
        )

    eval_df = pd.DataFrame(rows, columns=EVAL_COLUMNS)
    if eval_df.empty:
        return eval_df

    return eval_df.sort_values(
        ["benchmark", "model_family", "curriculum"]
    ).reset_index(drop=True)


In [4]:
eval_df = load_eval_table()
print(f"Eval rows: {len(eval_df)}")
eval_df

Eval rows: 96


,benchmark,model_family,curriculum,run_name,metric_name,metric_value,num_examples,checkpoint_path,model_id,metrics_path
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846416,1172,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.880546,1172,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887372,1172,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.896758,1172,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926000,1000,ckpt/qwen_sciq_frontloaded_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932000,1000,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928000,1000,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...


In [5]:
eval_df.groupby(["benchmark", "model_family"])["curriculum"].agg(list)

benchmark  model_family
arc        gpt_oss         [backloaded, fixed_k_1, fixed_k_max, frontload...
           lfm2            [backloaded, fixed_k_1, fixed_k_max, frontload...
           olmoe           [backloaded, fixed_k_1, fixed_k_max, frontload...
           qwen            [backloaded, fixed_k_1, fixed_k_max, frontload...
gsm8k      gpt_oss         [backloaded, fixed_k_1, fixed_k_max, frontload...
           lfm2            [backloaded, fixed_k_1, fixed_k_max, frontload...
           olmoe           [backloaded, fixed_k_1, fixed_k_max, frontload...
           qwen            [backloaded, fixed_k_1, fixed_k_max, frontload...
sciq       gpt_oss         [backloaded, fixed_k_1, fixed_k_max, frontload...
           lfm2            [backloaded, fixed_k_1, fixed_k_max, frontload...
           olmoe           [backloaded, fixed_k_1, fixed_k_max, frontload...
           qwen            [backloaded, fixed_k_1, fixed_k_max, frontload...
Name: curriculum, dtype: object

## 2. Query W&B Local

This section queries W&B only after the eval table is already extracted and visible.

Set these first if needed:

- `WANDB_BASE_URL`
- `WANDB_API_KEY`
- `WANDB_ENTITY`
- `WANDB_PROJECT`


In [6]:
if WANDB_ENTITY == "REPLACE_ME":
    raise ValueError(
        "Set WANDB_ENTITY before running the W&B cells. "
        "Example: export WANDB_ENTITY=your-team-or-username"
    )

api = wandb.Api(overrides={"base_url": WANDB_BASE_URL}, timeout=60)
WANDB_BASE_URL, WANDB_ENTITY, WANDB_PROJECT

wandb: Loading settings from /home/sebi/.config/wandb/settings
wandb: [wandb.Api()] Loaded credentials for http://localhost:8080 from /home/sebi/.netrc.


('http://localhost:8080', 'sebi', 'routing_curriculum')

In [7]:
def load_wandb_runtime_table(eval_run_names):
    rows = []

    eval_name_lookup = {}
    for eval_run_name in eval_run_names:
        eval_name_lookup[eval_run_name] = eval_run_name
        base_run_name, separator, eval_variant = eval_run_name.partition("__")
        if separator and eval_variant in {"best", "final"}:
            eval_name_lookup.setdefault(base_run_name, eval_run_name)

    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

    for run in runs:
        config = dict(run.config or {})
        summary = dict(run.summary or {})

        output_dir = config.get("output_dir")
        output_run_name = Path(output_dir).name if output_dir else None
        candidate_run_name = output_run_name or run.name

        matched_eval_run_name = eval_name_lookup.get(candidate_run_name) or eval_name_lookup.get(run.name)
        if matched_eval_run_name is None:
            continue

        total_runtime_seconds = summary.get("_runtime")
        if total_runtime_seconds is None:
            total_runtime_seconds = summary.get("_wandb", {}).get("runtime")
        if total_runtime_seconds is None:
            total_runtime_seconds = summary.get("train_runtime")

        rows.append(
            {
                "run_name": matched_eval_run_name,
                "train_run_name": candidate_run_name,
                "wandb_run_name": run.name,
                "wandb_run_id": run.id,
                "wandb_run_path": "/".join(run.path),
                "wandb_run_url": getattr(run, "url", None),
                "created_at": getattr(run, "created_at", None),
                "state": getattr(run, "state", None),
                "wandb_project": WANDB_PROJECT,
                "dataset_id": config.get("dataset_id"),
                "output_dir": output_dir,
                "total_runtime_seconds": total_runtime_seconds,
                "train_runtime_seconds": summary.get("train_runtime"),
            }
        )

    expected_columns = [
        "run_name",
        "train_run_name",
        "wandb_run_name",
        "wandb_run_id",
        "wandb_run_path",
        "wandb_run_url",
        "created_at",
        "state",
        "wandb_project",
        "dataset_id",
        "output_dir",
        "total_runtime_seconds",
        "train_runtime_seconds",
    ]

    wandb_df = pd.DataFrame(rows, columns=expected_columns)
    if wandb_df.empty:
        return wandb_df, pd.DataFrame(columns=["run_name", "num_wandb_matches"])

    duplicate_counts = (
        wandb_df.groupby("run_name")
        .size()
        .rename("num_wandb_matches")
        .reset_index()
        .query("num_wandb_matches > 1")
        .sort_values(["num_wandb_matches", "run_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

    wandb_df = wandb_df.sort_values(["created_at", "wandb_run_id"]).drop_duplicates(
        subset="run_name", keep="last"
    )

    return wandb_df.reset_index(drop=True), duplicate_counts


In [8]:
wandb_df, duplicate_wandb_runs = load_wandb_runtime_table(set(eval_df["run_name"]))
print(f"Matched W&B rows: {len(wandb_df)}")
wandb_df.sort_values(["run_name"]).reset_index(drop=True)

Matched W&B rows: 96


,run_name,train_run_name,wandb_run_name,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,total_runtime_seconds,train_runtime_seconds
0,gpt_oss_arc_backloaded_lora__final,gpt_oss_arc_backloaded_lora,gpt_oss_arc_backloaded_lora,wyc3wnvc,sebi/routing_curriculum/wyc3wnvc,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:17:37Z,finished,routing_curriculum,None,None,426.705014,301.1374
1,gpt_oss_arc_fixed_k_1_lora__final,gpt_oss_arc_fixed_k_1_lora,gpt_oss_arc_fixed_k_1_lora,eeal4khf,sebi/routing_curriculum/eeal4khf,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:10:37Z,finished,routing_curriculum,None,None,385.221270,266.4828
2,gpt_oss_arc_fixed_k_max_lora__final,gpt_oss_arc_fixed_k_max_lora,gpt_oss_arc_fixed_k_max_lora,cwn0oios,sebi/routing_curriculum/cwn0oios,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:02:19Z,finished,routing_curriculum,None,None,494.795154,356.9242
3,gpt_oss_arc_frontloaded_lora__final,gpt_oss_arc_frontloaded_lora,gpt_oss_arc_frontloaded_lora,hi2k7gtl,sebi/routing_curriculum/hi2k7gtl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:09:48Z,finished,routing_curriculum,None,None,466.192986,334.4224
4,gpt_oss_arc_jump_warmup_lora__final,gpt_oss_arc_jump_warmup_lora,gpt_oss_arc_jump_warmup_lora,276gyaym,sebi/routing_curriculum/276gyaym,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:40:29Z,finished,routing_curriculum,None,None,476.088488,351.1362
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,qwen_sciq_frontloaded_lora__final,qwen_sciq_frontloaded_lora,qwen_sciq_frontloaded_lora,mk5m1clx,sebi/routing_curriculum/mk5m1clx,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T18:11:50Z,finished,routing_curriculum,None,None,1835.730935,1730.1586
92,qwen_sciq_jump_warmup_lora__final,qwen_sciq_jump_warmup_lora,qwen_sciq_jump_warmup_lora,45x501yl,sebi/routing_curriculum/45x501yl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T20:13:18Z,finished,routing_curriculum,None,None,1892.638901,1786.6832
93,qwen_sciq_linear_k_1_to_topk_lora__final,qwen_sciq_linear_k_1_to_topk_lora,qwen_sciq_linear_k_1_to_topk_lora,lat4zy9p,sebi/routing_curriculum/lat4zy9p,http://localhost:8080/sebi/routing_curriculum/...,2026-04-16T20:46:52Z,finished,routing_curriculum,None,None,1773.550400,1663.2053
94,qwen_sciq_linear_mid_start_lora__final,qwen_sciq_linear_mid_start_lora,qwen_sciq_linear_mid_start_lora,k77qiuso,sebi/routing_curriculum/k77qiuso,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T19:11:10Z,finished,routing_curriculum,None,None,1821.679299,1717.7866


In [9]:
duplicate_wandb_runs

,run_name,num_wandb_matches


## 3. Join Eval Metrics with W&B Runtimes


In [10]:
comparison_table = eval_df.merge(
    wandb_df,
    on="run_name",
    how="left",
    validate="one_to_one",
)

print(f"Rows missing matched runtime: {comparison_table['total_runtime_seconds'].isna().sum()}")
comparison_table.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

Rows missing matched runtime: 0


,benchmark,model_family,curriculum,run_name,metric_name,metric_value,num_examples,checkpoint_path,model_id,metrics_path,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,total_runtime_seconds,train_runtime_seconds
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,wyc3wnvc,sebi/routing_curriculum/wyc3wnvc,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:17:37Z,finished,routing_curriculum,None,None,426.705014,301.1374
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846416,1172,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eeal4khf,sebi/routing_curriculum/eeal4khf,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:10:37Z,finished,routing_curriculum,None,None,385.221270,266.4828
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.880546,1172,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,cwn0oios,sebi/routing_curriculum/cwn0oios,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:02:19Z,finished,routing_curriculum,None,None,494.795154,356.9242
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887372,1172,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,hi2k7gtl,sebi/routing_curriculum/hi2k7gtl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:09:48Z,finished,routing_curriculum,None,None,466.192986,334.4224
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.896758,1172,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,276gyaym,sebi/routing_curriculum/276gyaym,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:40:29Z,finished,routing_curriculum,None,None,476.088488,351.1362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926000,1000,ckpt/qwen_sciq_frontloaded_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,mk5m1clx,sebi/routing_curriculum/mk5m1clx,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T18:11:50Z,finished,routing_curriculum,None,None,1835.730935,1730.1586
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932000,1000,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,45x501yl,sebi/routing_curriculum/45x501yl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T20:13:18Z,finished,routing_curriculum,None,None,1892.638901,1786.6832
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,lat4zy9p,sebi/routing_curriculum/lat4zy9p,http://localhost:8080/sebi/routing_curriculum/...,2026-04-16T20:46:52Z,finished,routing_curriculum,None,None,1773.550400,1663.2053
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928000,1000,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,k77qiuso,sebi/routing_curriculum/k77qiuso,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T19:11:10Z,finished,routing_curriculum,None,None,1821.679299,1717.7866


## 4. Compute Baseline Deltas

Baseline is the `fixed_k_max` run within each `(benchmark, model_family)` group.


In [11]:
def absolute_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline):
        return pd.NA
    return value - baseline


def pct_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline) or baseline == 0:
        return pd.NA
    return (value - baseline) / baseline * 100.0


BASELINE_CURRICULUM = "fixed_k_max"

baseline_table = (
    comparison_table.loc[
        comparison_table["curriculum"] == BASELINE_CURRICULUM,
        ["benchmark", "model_family", "metric_value", "total_runtime_seconds"],
    ]
    .rename(
        columns={
            "metric_value": "baseline_metric_value",
            "total_runtime_seconds": "baseline_total_runtime_seconds",
        }
    )
)

baseline_table

,benchmark,model_family,baseline_metric_value,baseline_total_runtime_seconds
2,arc,gpt_oss,0.880546,494.795154
10,arc,lfm2,0.842150,347.984252
18,arc,olmoe,0.677474,237.433350
26,arc,qwen,0.791809,415.023331
34,gsm8k,gpt_oss,0.714177,10865.041912
42,gsm8k,lfm2,0.598180,6923.694454
50,gsm8k,olmoe,0.366945,5130.713275
58,gsm8k,qwen,0.401061,7422.546616
66,sciq,gpt_oss,0.951000,2316.064000
74,sciq,lfm2,0.952000,1744.984367


In [12]:
comparison_table = comparison_table.merge(
    baseline_table,
    on=["benchmark", "model_family"],
    how="left",
    validate="many_to_one",
)

comparison_table["metric_delta"] = comparison_table.apply(
    lambda row: absolute_delta(row["metric_value"], row["baseline_metric_value"]),
    axis=1,
)
comparison_table["runtime_delta_pct"] = comparison_table.apply(
    lambda row: pct_delta(row["total_runtime_seconds"], row["baseline_total_runtime_seconds"]),
    axis=1,
)
comparison_table["is_baseline"] = comparison_table["curriculum"].eq(BASELINE_CURRICULUM)

comparison_table = comparison_table[
    [
        "benchmark",
        "model_family",
        "curriculum",
        "run_name",
        "metric_name",
        "metric_value",
        "baseline_metric_value",
        "metric_delta",
        "total_runtime_seconds",
        "baseline_total_runtime_seconds",
        "runtime_delta_pct",
        "is_baseline",
        "wandb_run_name",
        "wandb_run_id",
        "wandb_run_path",
        "wandb_run_url",
        "created_at",
        "state",
        "wandb_project",
        "dataset_id",
        "output_dir",
        "checkpoint_path",
        "metrics_path",
    ]
].sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

comparison_table

,benchmark,model_family,curriculum,run_name,metric_name,metric_value,baseline_metric_value,metric_delta,total_runtime_seconds,baseline_total_runtime_seconds,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,checkpoint_path,metrics_path
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876280,0.880546,-0.004266,426.705014,494.795154,...,wyc3wnvc,sebi/routing_curriculum/wyc3wnvc,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:17:37Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846416,0.880546,-0.034130,385.221270,494.795154,...,eeal4khf,sebi/routing_curriculum/eeal4khf,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:10:37Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.880546,0.880546,0.000000,494.795154,494.795154,...,cwn0oios,sebi/routing_curriculum/cwn0oios,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:02:19Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887372,0.880546,0.006826,466.192986,494.795154,...,hi2k7gtl,sebi/routing_curriculum/hi2k7gtl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:09:48Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.896758,0.880546,0.016212,476.088488,494.795154,...,276gyaym,sebi/routing_curriculum/276gyaym,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:40:29Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926000,0.929000,-0.003000,1835.730935,1920.494960,...,mk5m1clx,sebi/routing_curriculum/mk5m1clx,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T18:11:50Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_frontloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/sciq/qwen_s...
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932000,0.929000,0.003000,1892.638901,1920.494960,...,45x501yl,sebi/routing_curriculum/45x501yl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T20:13:18Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,/home/sebi/disertatie/eval_results/sciq/qwen_s...
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927000,0.929000,-0.002000,1773.550400,1920.494960,...,lat4zy9p,sebi/routing_curriculum/lat4zy9p,http://localhost:8080/sebi/routing_curriculum/...,2026-04-16T20:46:52Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,/home/sebi/disertatie/eval_results/sciq/qwen_s...
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928000,0.929000,-0.001000,1821.679299,1920.494960,...,k77qiuso,sebi/routing_curriculum/k77qiuso,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T19:11:10Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,/home/sebi/disertatie/eval_results/sciq/qwen_s...


In [13]:
display_table = comparison_table.copy()
for column in [
    "metric_value",
    "baseline_metric_value",
    "metric_delta",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(3)

for column in [
    "total_runtime_seconds",
    "baseline_total_runtime_seconds",
    "runtime_delta_pct",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(2)

display_table

,benchmark,model_family,curriculum,run_name,metric_name,metric_value,baseline_metric_value,metric_delta,total_runtime_seconds,baseline_total_runtime_seconds,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,checkpoint_path,metrics_path
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876,0.881,-0.004,426.71,494.80,...,wyc3wnvc,sebi/routing_curriculum/wyc3wnvc,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:17:37Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846,0.881,-0.034,385.22,494.80,...,eeal4khf,sebi/routing_curriculum/eeal4khf,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:10:37Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.881,0.881,0.000,494.80,494.80,...,cwn0oios,sebi/routing_curriculum/cwn0oios,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:02:19Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887,0.881,0.007,466.19,494.80,...,hi2k7gtl,sebi/routing_curriculum/hi2k7gtl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:09:48Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.897,0.881,0.016,476.09,494.80,...,276gyaym,sebi/routing_curriculum/276gyaym,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:40:29Z,finished,routing_curriculum,None,None,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926,0.929,-0.003,1835.73,1920.49,...,mk5m1clx,sebi/routing_curriculum/mk5m1clx,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T18:11:50Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_frontloaded_lora/final_adapter,/home/sebi/disertatie/eval_results/sciq/qwen_s...
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932,0.929,0.003,1892.64,1920.49,...,45x501yl,sebi/routing_curriculum/45x501yl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T20:13:18Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,/home/sebi/disertatie/eval_results/sciq/qwen_s...
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927,0.929,-0.002,1773.55,1920.49,...,lat4zy9p,sebi/routing_curriculum/lat4zy9p,http://localhost:8080/sebi/routing_curriculum/...,2026-04-16T20:46:52Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,/home/sebi/disertatie/eval_results/sciq/qwen_s...
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928,0.929,-0.001,1821.68,1920.49,...,k77qiuso,sebi/routing_curriculum/k77qiuso,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T19:11:10Z,finished,routing_curriculum,None,None,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,/home/sebi/disertatie/eval_results/sciq/qwen_s...


In [14]:
reduced_table = display_table[[
    "benchmark",
    "model_family",
    "curriculum",
    "metric_value",
    "metric_delta",
    "total_runtime_seconds",
    "runtime_delta_pct",
]]

reduced_table["metric_value"] = reduced_table.apply(
    lambda row: (
        f"{row['metric_value']:.3f} ({row['metric_delta']:+.3f})"
        if (row["curriculum"] != BASELINE_CURRICULUM and not pd.isna(row['metric_delta']))
        else f"{row['metric_value']:.3f}"
    ),
    axis=1,
)

reduced_table["total_runtime_seconds"] = reduced_table.apply(
    lambda row: (
        f"{row['total_runtime_seconds']}s ({row['runtime_delta_pct']:+.2f}%)"
        if (row["curriculum"] != BASELINE_CURRICULUM and not pd.isna(row['runtime_delta_pct']))
        else f"{row['total_runtime_seconds']}s"
    ),
    axis=1,
)

reduced_table = reduced_table.drop(columns=["metric_delta", "runtime_delta_pct"])

reduced_table

curriculum_rank = {name: index for index, name in enumerate(CURRICULUM_DISPLAY_ORDER)}
reduced_table["curriculum_sort_key"] = reduced_table["curriculum"].map(
    lambda value: curriculum_rank.get(value, len(curriculum_rank))
)
reduced_table = reduced_table.sort_values(
    ["benchmark", "model_family", "curriculum_sort_key", "curriculum"]
).drop(columns=["curriculum_sort_key"]).reset_index(drop=True)
reduced_table

,benchmark,model_family,curriculum,metric_value,total_runtime_seconds
0,arc,gpt_oss,fixed_k_max,0.881,494.8s
1,arc,gpt_oss,fixed_k_1,0.846 (-0.034),385.22s (-22.15%)
2,arc,gpt_oss,linear_k_1_to_topk,0.876 (-0.004),438.95s (-11.29%)
3,arc,gpt_oss,frontloaded,0.887 (+0.007),466.19s (-5.78%)
4,arc,gpt_oss,backloaded,0.876 (-0.004),426.71s (-13.76%)
...,...,...,...,...,...
91,sciq,qwen,frontloaded,0.926 (-0.003),1835.73s (-4.41%)
92,sciq,qwen,backloaded,0.928 (-0.001),1718.15s (-10.54%)
93,sciq,qwen,warmup,0.922 (-0.007),1900.79s (-1.03%)
94,sciq,qwen,jump_warmup,0.932 (+0.003),1892.64s (-1.45%)


In [15]:
curriculum_rank = {name: index for index, name in enumerate(CURRICULUM_DISPLAY_ORDER)}
available_curricula = sorted(
    set(reduced_table["curriculum"].dropna()),
    key=lambda value: (curriculum_rank.get(value, len(curriculum_rank)), value),
)

CURRICULUM_LABELS = {
    "fixed_k_max": "Fixed K Default",
}


def curriculum_label(curriculum):
    return CURRICULUM_LABELS.get(curriculum, curriculum.replace("_", " ").title())


metric_table = reduced_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="metric_value",
)
metric_table = metric_table.reindex(columns=available_curricula)
metric_table.columns = [curriculum_label(curriculum) for curriculum in metric_table.columns]

runtime_table = reduced_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="total_runtime_seconds",
)
runtime_table = runtime_table.reindex(columns=available_curricula)
runtime_table.columns = [curriculum_label(curriculum) for curriculum in runtime_table.columns]

delta_table = comparison_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values=["metric_delta", "runtime_delta_pct"],
)
delta_table = delta_table.reindex(
    columns=pd.MultiIndex.from_product(
        [["metric_delta", "runtime_delta_pct"], available_curricula]
    )
)
delta_table = delta_table.swaplevel(0, 1, axis=1)

metric_delta_table = delta_table.xs("metric_delta", axis=1, level=1)
metric_delta_table.columns = [curriculum_label(curriculum) for curriculum in metric_delta_table.columns]

runtime_delta_table = delta_table.xs("runtime_delta_pct", axis=1, level=1)
runtime_delta_table.columns = [curriculum_label(curriculum) for curriculum in runtime_delta_table.columns]


def highlight_metric_cells(row):
    styles = pd.Series("", index=row.index)
    delta_row = metric_delta_table.loc[row.name]

    for column in row.index:
        if column == curriculum_label(BASELINE_CURRICULUM):
            continue

        delta_value = delta_row.get(column)
        if pd.isna(delta_value) or delta_value == 0:
            continue

        color = "#56ad34" if delta_value > 0 else "#d65f5f"
        styles[column] = f"background-color: {color}; font-weight: 600"

    return styles


def highlight_runtime_cells(row):
    styles = pd.Series("", index=row.index)
    delta_row = runtime_delta_table.loc[row.name]

    for column in row.index:
        if column == curriculum_label(BASELINE_CURRICULUM):
            continue

        delta_value = delta_row.get(column)
        if pd.isna(delta_value) or delta_value == 0:
            continue

        color = "#578aba" if delta_value < 0 else "#e6c84f"
        styles[column] = f"background-color: {color}; font-weight: 600"

    return styles


(
    metric_table.style.set_caption("Grouped eval metric deltas")
    .apply(highlight_metric_cells, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)


In [ ]:
(
    runtime_table.style.set_caption("Grouped eval runtime deltas")
    .apply(highlight_runtime_cells, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)